In [1]:
from llama_index.core import KnowledgeGraphIndex, SimpleDirectoryReader
from llama_index.core import StorageContext
from llama_index.graph_stores.neo4j import Neo4jGraphStore


from llama_index.llms.openai import OpenAI
from IPython.display import Markdown, display

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
# from fastapi import FastAPI
from llama_index.core import StorageContext, load_index_from_storage
from custom_llm import OurLLM
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer

llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))

[nltk_data] Downloading package punkt_tab to c:\Users\M LAKSHMI
[nltk_data]     NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-
[nltk_data]     packages\llama_index\core\_static/nltk_cache...
[nltk_data]   Package punkt_tab is already up-to-date!
c:\Users\M LAKSHMI NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_4156\3885403216.py:26: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))


In [2]:
import os
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
NEO4J_DATABASE = "test"

graph_store = Neo4jGraphStore(
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    url=NEO4J_URI,
    database=NEO4J_DATABASE,
)

storage_context = StorageContext.from_defaults(graph_store=graph_store)

In [3]:
# Clean dataset first
ew = graph_store.query(
    """
MATCH (n) detach delete n
"""
)

In [4]:
file = 'balakanda.json'
import json
with open(file) as f:
    data = json.load(f)

from llama_index.core import Document
documents = []
for i in data[:]:       
    document = Document(
    text = i['Prompt_Answer'],
    metadata={
        "canto_summary": i['Summary'],
        "r_id": str(i['Canto']) + "_" + str(i['Prompt_Number'])
    },
    excluded_llm_metadata_keys=["r_id"],
    metadata_seperator="::",
    metadata_template="{value}",
    text_template="Content: {content}, Chapter context: {metadata_str}\n",
    )
    documents.append(document)
print(len(documents))
print(f"Document Metadata: {documents[-1].metadata}")
storage_context = StorageContext.from_defaults(graph_store=graph_store)

JSONDecodeError: Invalid control character at: line 440 column 99 (char 69045)

In [ ]:
from llama_index.core.prompts.base import PromptTemplate
from llama_index.core.prompts.prompt_type import PromptType
DEFAULT_KG_TRIPLET_EXTRACT_TMPL_NEW = (
    "Some text from ancient story Ramayan is provided below. Given the text, extract knowledge graph triplets with the following rules:\n"

    "1. Extract triplets with the following patterns:\n"
    "   (charecter, relation, charecter) - Resolve the relations between charecters based on the context given\n"
    "   (charecter, context, event) - Resolve the relations between charecters and the major events in the context given\n"
    "   (charecter, 'is in', Canto) - Mapping for each charecter to canto\n"
    "2. Focus more on the important events and relations between charecters\n"
    "3. Avoid stopwords. The output should only be triplets and no extra commenting/template literals.\n"
    "4. use snake casing for everything in triplets\n"
    "5. for (charecter, relation, charecter) type of triplets, use the following relations:\n"
    "- is (mother/father/brother/sister/wife/husband/son/daughter/siblings/aunt/uncle/niece/nephew/teacher/mentor/related_in_canto_'number') of\n"
    "6. While generating triplets corresponding to charecter context event, focus only on important event of the given text, no need to split every sentence of text\n"
    "7. The number of triplets triplets of type (charecter, 'is in', Canto) should not exceed 10, and the number of other two type of triplets should not exceed {max_knowledge_triplets} \n"
    "These triplets are used for RAG, so only give important triplets and make sure there is no important data missing\n"
    "Text: {text}\n"
    "Triplets:\n"
)
DEFAULT_KG_TRIPLET_EXTRACT_PROMPT_NEW = PromptTemplate(
    DEFAULT_KG_TRIPLET_EXTRACT_TMPL_NEW,
    prompt_type=PromptType.KNOWLEDGE_TRIPLET_EXTRACT,
)

In [ ]:
index = KnowledgeGraphIndex.from_documents(
    documents,
    kg_triplet_extract_template = DEFAULT_KG_TRIPLET_EXTRACT_PROMPT_NEW,
    storage_context=storage_context,
    max_triplets_per_chunk=20,
    include_embeddings=True
)

In [ ]:
# count blank lines in log.txt
with open('log.txt', 'r') as f:
    lines = f.readlines()
    count = 0
    for line in lines:
        if line == '\n':
            count += 1
    print(count)
 

In [ ]:
file = 'balakanda.json'
import json
with open(file) as f:
    data = json.load(f)

from llama_index.core import Document
documents = []
for i in data[:]:       
    document = Document(
    text = i['Prompt_Answer'],
    metadata={
        "canto_summary": i['Summary'],
        "r_id": str(i['Canto']) + "_" + str(i['Prompt_Number'])
    },
    excluded_llm_metadata_keys=["r_id"],
    metadata_seperator="::",
    metadata_template="{value}",
    text_template="Content: {content}, Chapter context: {metadata_str}\n",
    )
    documents.append(document)
print(len(documents))
print(f"Document Metadata: {documents[-1].metadata}")
storage_context = StorageContext.from_defaults(graph_store=graph_store)

In [ ]:
index2 = KnowledgeGraphIndex.from_documents(
    documents[219:],
    kg_triplet_extract_template = DEFAULT_KG_TRIPLET_EXTRACT_PROMPT_NEW,
    storage_context=storage_context,
    max_triplets_per_chunk=20,
    include_embeddings=True
)

In [ ]:
documents[230].text


In [ ]:
index2.storage_context.persist(persist_dir="test_storage")

In [5]:
from llama_index.core import StorageContext, load_index_from_storage, load_indices_from_storage
storage_context2 = StorageContext.from_defaults(persist_dir="storage")
indexi = load_index_from_storage(storage_context2)

In [6]:
query_engine = indexi.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=5,
    verbose=True)
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))

C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_4156\1842048155.py:7: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))


In [9]:
response = query_engine.query("Who are valmiki's students")
print(response)

Prompt num:  5
Extracted keywords: ['learners', 'students', 'followers', 'valmiki', 'disciples']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('valmiki', 'is_teacher_of', 'bharadwaja')
('valmiki', 'instructs', 'canto_4')
('valmiki', 'teacher_of', 'kusa')
('valmiki', 'teacher_of', 'lava')
('valmiki', 'taught', 'kusa')
Prompt num:  6
Kusa, Lava, and Bharadwaja are Valmiki's students.  Kusa and Lava were taught the Ramayana by Valmiki, and Bharadwaja is identified as Valmiki's disciple.



In [5]:
response = query_engine.query("who are wife of rishyasringa")
print(response)

Prompt num:  5
Extracted keywords: ['rishi', 'sanskrit', 'shanta', 'ramayan', 'wife', 'rishyasringa']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('rishyasringa', 'has', 'spiritual_power')
('rishyasringa', 'husband_of', 'santa')
('sita', 'is_wife_of', 'rama')
('sita', 'wife_of', 'rama')
('rama', 'wife_of', 'sita')
Prompt num:  6
Santa is the wife of Rishyasringa.



In [6]:
response = query_engine.query("who are parents of santa")
print(response)

Prompt num:  7
Extracted keywords: ['santa', 'parents']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('santa', 'is_adopted_daughter_of', 'romapada')
('santa', 'is_daughter_of', 'romapada')
('romapada', 'is_father_of', 'santa')
('santa', 'daughter_of', 'romapada')
('romapada', 'father_of', 'santa')
Prompt num:  8
Dasaratha and an unnamed mother.  Santa was adopted by Romapada.



In [8]:
response = query_engine.query("who pro|phosised that sage rishyasringa will give birth to dasaratha's son")
print(response)

Prompt num:  11
Extracted keywords: ['birth', 'Rishyasringa', 'son', 'prophecy', 'Dasaratha']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('king_dasaratha', 'anticipated_birth_of', 'sons')
('king_dasaratha', 'is_father_of', 'rama')
('dasaratha', 'is_father_of', 'rama')
('dasaratha', 'father_of', 'vishnu')
('dasaratha', 'father_of', 'rama')
Prompt num:  12
The provided text doesn't mention a prophecy about Rishyasringa giving birth to Dasaratha's sons. It states that Rishyasringa performed sacrifices for Dasaratha, after which the king returned to Ayodhya and his queens gave birth to his sons.



In [9]:
response = query_engine.query("name of viswamitra's cow")
print(response)

Prompt num:  13
Extracted keywords: ['Viswamitra', 'name', 'cow']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('viswamitra', 'offered_cows_for', 'sabala')
('viswamitra', 'is_in', 'canto_58')
('viswamitra', 'is_in', 'canto_22')
('viswamitra', 'urged', 'animals')
('viswamitra', 'offers', 'cows')
Prompt num:  14
Viswamitra did not own the cow. The cow, Sabala, belonged to Vasistha.  Viswamitra desired Sabala and tried to obtain her through offers, force, and even war, but ultimately failed.



In [11]:
response = query_engine.query("which canto rama is avatar of vishnu")
print(response)

Prompt num:  16
Extracted keywords: ['Rama', 'Vishnu', 'Avatar', 'Canto']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('vishnu', 'is_in', 'canto_xvii')
('vishnu', 'is_in', 'canto_xvi')
('vishnu', 'is_in', 'canto_xv')
('vishnu', 'is_in', 'canto_40')
('vishnu', 'is_in', 'canto_45')
Prompt num:  17
Canto XV describes Vishnu agreeing to be born as Dasaratha's son (Rama) to defeat Ravana.  Therefore, the answer is Canto XV.



In [12]:
response = query_engine.query("when is it revealed that rama is avatar of vishnu")
print(response)

Prompt num:  18
Extracted keywords: ['revealed', 'Rama', 'Vishnu', 'avatar']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('rama', 'incarnation_of', 'vishnu')
('vishnu', 'is_in', 'canto_xvii')
('vishnu', 'is_in', 'canto_xvi')
('vishnu', 'is_in', 'canto_xv')
('vishnu', 'is_in', 'canto_45')
Prompt num:  19
Canto XXIX reveals that Rama is an incarnation of Vishnu.  This occurs when Viswamitra leads Rama and Lakshmana into the hermitage of Siddhagrama, explaining that it belongs to Rama as much as himself because Rama is Vishnu's incarnation.



In [13]:
query_engine = indexi.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=10,
    verbose=True)
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))
response = query_engine.query("when is it revealed that rama is avatar of vishnu")
print(response)

C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_14244\1369925528.py:7: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))


Prompt num:  20
Extracted keywords: ['revealed', 'Rama', 'Vishnu', 'avatar']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('vishnu', 'decided_human_form', 'canto_xvi')
('vishnu', 'incarnate_as', 'dasaratha_son')
('rama', 'incarnation_of', 'vishnu')
('vishnu', 'is_in', 'canto_lxii')
('vishnu', 'is_in', 'canto_xxix')
('vishnu', 'is_in', 'canto_xvii')
('vishnu', 'is_in', 'canto_xvi')
('vishnu', 'is_in', 'canto_xv')
('vishnu', 'is_in', 'canto_40')
('vishnu', 'is_in', 'canto_45')
Prompt num:  21
Canto XXIX reveals that Rama is an incarnation of Vishnu.  This is when Viswamitra leads Rama and Lakshmana into Siddhagrama, explaining that it belongs to Rama as much as to himself *because* Rama is a Vishnu incarnation.



In [14]:
response = query_engine.query("when did vishnu promise to reincarnate as rama")
print(response)

Prompt num:  22
Extracted keywords: ['reincarnate', 'Vishnu', 'promise', 'Rama', 'incarnation']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('gods', 'context_canto_xv', 'request_vishnu_incarnate')
('vishnu', 'secured', 'nectar_of_immortality')
('vishnu', 'decided_human_form', 'canto_xvi')
('rama', 'life_narrated_in', 'ramayana')
('rama', 'incarnation_of', 'vishnu')
('vishnu', 'is_in', 'canto_xvii')
('vishnu', 'is_in', 'canto_xvi')
('vishnu', 'is_in', 'canto_xv')
('vishnu', 'is_in', 'canto_40')
('vishnu', 'is_in', 'canto_45')
Prompt num:  23
Vishnu promised to reincarnate during the sacrifice performed by King Dasaratha for obtaining sons (Canto XV).  The gods approached Brahma and Vishnu during this sacrifice, requesting Vishnu's intervention to defeat Ravana. Vishnu then agreed to be born as Dasaratha's son.



In [16]:
response = query_engine.query("which sage prophosised that dasaratha will have sons")
print(response)

Prompt num:  26
Extracted keywords: ['prophesy', 'Dasaratha', 'sons', 'Sage']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('king_dasaratha', 'anticipated_birth_of', 'sons')
('king_dasaratha', 'will_have', 'four_sons')
('dasaratha', 'is_father_of', 'shatrughna')
('king_dasaratha', 'is_father_of', 'rama')
('dasaratha', 'father_of', 'shatrughna')
('king_dasaratha', 'father_of', 'rama')
('dasaratha', 'is_father_of', 'rama')
('dasaratha', 'father_of', 'vishnu')
('rama', 'son_of', 'king_dasaratha')
('dasaratha', 'father_of', 'rama')
Prompt num:  27
Rishyasringa.



In [17]:
response = query_engine.query("how many sons does king sagara have")
print(response)

Prompt num:  28
Extracted keywords: ['Sagara', 'sons', 'number', 'King', 'King Sagara']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('sagara', 'related_in_canto_38', 'sixty_thousand_sons')
('sagara_sons', 'related_in_canto_40', 'mahapadma')
('sagara', 'is_father_of', 'sixty_thousand_sons')
('sagara', 'father_of', 'sixty_thousand_sons')
('sagara_sons', 'sons_of_sagara', 'canto_xl')
('sixty_thousand_sons', 'sons_of', 'sagara')
('sagara', 'is_father_of', 'sons_of_sagara')
('sagara', 'is_father_of', 'sagara_sons')
('sons_of_sagara', 'is_in', 'canto_44')
('sagara', 'father_of', 'eldest_son')
Prompt num:  29
King Sagara had sixty thousand sons, plus one additional son named Asamanjas, for a total of sixty thousand and one.



In [18]:
response = query_engine.query("how many sons does viswamitra lose in battle")
print(response)

Prompt num:  30
Extracted keywords: ['battle', 'sons', 'killed', 'war', 'number', 'lose', 'death', 'Viswamitra']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('viswamitra', 'witnessed_downfall_of', 'vasistha_sons')
('viswamitra', 'related_in_canto_lix', 'vasistha_sons')
('viswamitra', 'army_destroyed', 'sabala_sons')
('viswamitra', 'father_of', 'viswamitra_sons')
('viswamitra_sons', 'sons_of', 'viswamitra')
('viswamitra', 'cursed', 'vasistha_sons')
('viswamitra', 'sons_death', 'canto_lv')
('viswamitra', 'father_of', 'maharatha')
('viswamitra', 'instructed', 'sons')
('viswamitra', 'has', 'sons')
Prompt num:  31
Viswamitra loses one hundred sons in battle against Vasistha.  While he had more than four sons (Havispanda, Madhuspanda, Drdhanetra, and Maharatha are named), the text specifies that one hundred of his sons were killed in this particular conflict.

In [19]:
response = query_engine.query("who told valmiki ramayana for the first time")
print(response)

Prompt num:  32
Extracted keywords: ['narrated', 'told', 'time', 'first time', 'Valmiki', 'Ramayana', 'first']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('valmiki', 'brahma_visit', 'tasked_to_write_ramayana')
('rama', 'listened_to_ramayana', 'ramayana_recitation')
('rama', 'meeting_hanuman_and_sugriva', 'canto_1')
('valmiki', 'received_discourse_from', 'narada')
('rama', 'life_narrated_in', 'ramayana')
('rama', 'listens_ramayana', 'canto_4')
('valmiki', 'wrote_story_of', 'rama')
('rama', 'told_story_to', 'sugriva')
('valmiki', 'composed', 'ramayana')
('valmiki', 'wrote', 'ramayana')
Prompt num:  33
Narada is the one who first told Valmiki the story of Rama, inspiring him to compose the Ramayana.



In [13]:
response = query_engine.query("what part of payesam was eaten by Kaikeyi")
print(response)

Prompt num:  13
Extracted keywords: ['Kaikeyi', 'Payasam', 'eaten']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('kaikeyi', 'receives_divine_pudding', 'canto_xvi')
('kaikeyi', 'requests_coronation', 'kingship')
('kaikeyi', 'is_in', 'canto_viii')
('kaikeyi', 'is_in', 'canto_xvi')
('kaikeyi', 'caused', 'exile')
Prompt num:  14
Kaikeyi received a portion of the divine payasam (pudding).  Initially, she received half of the remaining portion after Kausalya and Sumitra's first shares.  Then, Dasaratha gave the final remaining portion to Sumitra, meaning Kaikeyi did *not* receive the largest share.



In [ ]:
response = query_engine.query("what is Rama's bow name ?")
print(response)

Prompt num:  15
Extracted keywords: ['name', 'bow', 'Rama']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('rama', 'participates_in', 'bow_breaking_ceremony')
('rama', "broke_shiva's_bow", 'canto_68')
('rama', 'context', 'bow_sacrifice')
('rama', 'breaks', 'bow')
('rama', 'broke', 'bow')
Prompt num:  16
The provided texts don't name Rama's bow. They describe Rama breaking *Shiva's bow* in the context of Sita's marriage contest.  While Rama used a bow during his journey with Viswamitra to protect the sage's sacrifices, that bow isn't named.



: 